# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
# TODO: Import the necessary libs
# For example: 
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool

In [3]:
# TODO: Load environment variables
import os
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
! pip install langchain-core langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 127.7 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.73.0
    Uninstalling openai-1.73.0:
      Successfully uninstalled openai-1.73.0

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
# TODO: Create retrieve_game tool 
# # It should use chroma client and collection you created 
# # chroma_client = chromadb.PersistentClient(path="chromadb") # collection = chroma_client.get_collection("udaplay") 
# # Tool Docstring: 
# # Semantic search: Finds most results in the vector DB 
# # args: # - query: a question about game industry. 
#  # You'll receive results as list. Each element contains: 
# # - Platform: like Game Boy, Playstation 5, Xbox 360...) 
# # - Name: Name of the Game # - YearOfRelease: Year when that game was released for that platform 
# # - Description: Additional details about the game
import os
import chromadb
from chromadb.utils import embedding_functions
from langchain_core.tools import tool

# Initialize Chroma client
chroma_client = chromadb.PersistentClient(path="chromadb")

embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL")
)

collection = chroma_client.get_collection(
    "udaplay",
    embedding_function=embedding_fn
)

@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search: Finds most results in the vector DB

    args:
    - query: a question about game industry.

    Returns:
    List of games with:
    - Platform
    - Name
    - YearOfRelease
    - Description
    """

    results = collection.query(
        query_texts=[query],
        n_results=5
    )

    games = []

    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]

        games.append({
            "Name": metadata.get("Name"),
            "Platform": metadata.get("Platform"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": doc
        })

    return games

#### Evaluate Retrieval Tool

In [6]:
# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result
from pydantic import BaseModel
from openai import OpenAI
import os
from langchain_core.tools import tool

# Define schema
class EvaluationReport(BaseModel):
    useful: bool
    description: str


@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    """
    Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.

    args:
    - question: original question from user
    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database

    Returns:
    - useful: whether the documents are useful to answer the question
    - description: explanation of the evaluation result
    """

    client = OpenAI(
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url=os.getenv("OPENAI_BASE_URL")
    )

    prompt = (
        f"Your task is to evaluate if the documents are enough to respond to the query. "
        f"Give a detailed explanation so it's possible to decide whether to accept them or not.\n\n"
        f"Question: {question}\n\n"
        f"Retrieved Documents:\n{retrieved_docs}"
    )

    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format=EvaluationReport
    )

    result = response.choices[0].message.parsed

    return {
        "useful": result.useful,
        "description": result.description
    }

#### Game Web Search Tool

In [7]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 
from tavily import TavilyClient
@tool
def game_web_search(question: str) -> dict:
    """
    Semantic search: Finds most results in the vector DB
    args:
    - question: a question about game industry.
    """
    tavily_client = TavilyClient(api_key=TAVILY_API_KEY)
    return tavily_client.search(question)


### Agent

In [8]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed
from lib.tooling import Tool
from lib.agents import Agent

tools = [
    Tool(retrieve_game.func),
    Tool(evaluate_retrieval.func),
    Tool(game_web_search.func)
  ]

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
         "You are UdaPlay, an AI research assistant for the video game industry.\n"
      "You MUST always follow this exact tool order:\n"
      "STEP 1: Always call retrieve_game first to search the local database.\n"
      "STEP 2: Always call evaluate_retrieval with the question and retrieved docs.\n"
      "STEP 3: Only call game_web_search if evaluate_retrieval returns useful=False.\n"
      "Never skip steps. Never call web search before trying retrieve and evaluate first."
      ),
    tools=tools
  )

In [9]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?
queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?"
  ]

for query in queries:
    print(f"\nQ: {query}")
    run = agent.invoke(query)
    print(f"A: {run.get_final_state()['messages'][-1].content}")


Q: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
A: Pokémon Gold and Silver were released in 1999.

Q: Which one was the first 3D platformer Mario game?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
A: The first 3D platformer Mario game is **Super Mario 64**, which was released in 1996.

Q: Was Mortal Kombat X released for PlayStation 5?
[StateMachin

In [10]:
from pydantic import BaseModel

class GameAnswer(BaseModel):
    answer: str
    source: str  # "vector_db" or "web_search"
    confidence: str  # "high", "medium", or "low"

  #Then when printing results, parse the final message into this structure by adding another LLM call:

from openai import OpenAI

def get_structured_answer(raw_answer: str, query: str) -> GameAnswer:
    client = OpenAI(
        api_key=OPENAI_API_KEY,
        base_url=os.getenv("OPENAI_BASE_URL")
      )
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": f"Query: {query}\nAnswer: {raw_answer}\nStructure this response."}
          ],
        response_format=GameAnswer
      )
    return response.choices[0].message.parsed

  # Use it like this:
for query in queries:
    run = agent.invoke(query)
    raw = run.get_final_state()["messages"][-1].content
    structured = get_structured_answer(raw, query)
    print(f"\nQ: {query}")
    print(f"A: {structured.answer}")
    print(f"Source: {structured.source}")
    print(f"Confidence: {structured.confidence}")

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Q: When was Pokémon Gold and Silver released?
A: Pokémon Gold and Silver were released in 1999.
Source: Pokémon Official Website
Confidence: High
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Q: Which one was the first 3D platformer Mario game?
A: The first 3D platformer Mario game is **Super Mario 64**, which was released in 1996.
Source: https://en.wikipedia.org/wiki/Super_Mario_64
Confidence: High
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Q: Was Mortal Kombat X released for PlayStation 5?
A: Mortal Kombat X is playable on PlayStation 5, but it was not specifically release

### (Optional) Advanced

In [23]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes

import json
from pathlib import Path
from lib.tooling import Tool
from lib.state_machine import StateMachine, Step, EntryPoint, Termination
from lib.agents import AgentState
from lib.messages import SystemMessage, UserMessage, AIMessage, ToolMessage
from lib.llm import LLM


class LongTermMemory:
    def __init__(self, path: str = "memory.json"):
        self.path = Path(path)
        self.data = self._load()

    def _load(self):
        if self.path.exists():
            return json.loads(self.path.read_text())
        return []

    def save(self, query: str, answer: str):
        self.data.append({"query": query, "answer": answer})
        self.path.write_text(json.dumps(self.data, indent=2))

    def get_context(self) -> str:
        if not self.data:
            return ""
        entries = [f"Q: {e['query']}\nA: {e['answer']}" for e in self.data[-5:]]
        return "\n\n".join(entries)


long_term_memory = LongTermMemory()

tools_map = {
    "retrieve_game": retrieve_game.func,
    "evaluate_retrieval": evaluate_retrieval.func,
    "game_web_search": game_web_search.func
}

lib_tools = [Tool(f, name=n) for n, f in tools_map.items()]
llm = LLM(model="gpt-4o-mini", tools=lib_tools)


def prepare_messages(state: AgentState) -> AgentState:
    memory_context = long_term_memory.get_context()
    instructions = (
          "You are UdaPlay, an AI research assistant for the video game industry.\n"
          "You MUST always follow this exact tool order:\n"
          "STEP 1: Always call retrieve_game first to search the local database.\n"
          "STEP 2: Always call evaluate_retrieval with the question and retrieved docs.\n"
          "STEP 3: Only call game_web_search if evaluate_retrieval returns useful=False.\n"
          "Never skip steps. Never call web search before trying retrieve and evaluate first.\n"
          "IMPORTANT: When citing sources, only use URLs that appear in the web search tool results. Do not generate or invent URLs."
      )
    if memory_context:
        instructions += f"\nPrevious knowledge:\n{memory_context}"

    messages = [SystemMessage(content=instructions)]

      # Carry over previous turns (user questions + AI answers only, skip tool messages)
    for msg in state.get("messages", []):
        if isinstance(msg, (UserMessage, AIMessage)) and not msg.content.startswith("[Tool:"):
            messages.append(msg)

    messages.append(UserMessage(content=state["user_query"]))
    return {"messages": messages}


def llm_step(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    tool_calls = response.tool_calls if response.tool_calls else None

    return {
        "messages": state["messages"] + [
            AIMessage(content=response.content, tool_calls=tool_calls)
        ],
        "current_tool_calls": tool_calls
    }


def retrieve_game_step(state: AgentState) -> AgentState:
    result = retrieve_game.func(query=state["user_query"])

    return {
        "messages": state["messages"] + [
            UserMessage(content=f"[Tool: retrieve_game]\n{str(result)}")
        ],
        "current_tool_calls": None
    }


def evaluate_retrieval_step(state: AgentState) -> AgentState:
    retrieved_docs = None

    for msg in reversed(state["messages"]):
        if isinstance(msg, UserMessage) and msg.content.startswith("[Tool: retrieve_game]"):
            retrieved_docs = msg.content.split("\n", 1)[1]
            break

    result = evaluate_retrieval.func(
        question=state["user_query"],
        retrieved_docs=retrieved_docs
    )

    return {
        "messages": state["messages"] + [
            UserMessage(content=f"[Tool: evaluate_retrieval]\n{str(result)}")
        ],
        "current_tool_calls": None
    }


def web_search_step(state: AgentState) -> AgentState:
    result = game_web_search.func(question=state["user_query"])

    return {
        "messages": state["messages"] + [
            UserMessage(content=f"[Tool: game_web_search]\n{str(result)}")
        ],
        "current_tool_calls": None
    }


def should_web_search(state: AgentState):
    for msg in reversed(state["messages"]):
        if isinstance(msg, UserMessage) and msg.content.startswith("[Tool: evaluate_retrieval]"):
            try:
                content = msg.content.split("\n", 1)[1]
                result = eval(content)
                if not result.get("useful"):
                    return web_search_node
            except:
                pass

    return llm_node


machine = StateMachine(AgentState)

entry = EntryPoint[AgentState]()
prep_node = Step[AgentState]("prepare_messages", prepare_messages)
llm_node = Step[AgentState]("llm", llm_step)
retrieve_node = Step[AgentState]("retrieve_game", retrieve_game_step)
evaluate_node = Step[AgentState]("evaluate_retrieval", evaluate_retrieval_step)
web_search_node = Step[AgentState]("game_web_search", web_search_step)
termination = Termination[AgentState]()

machine.add_steps([
    entry,
    prep_node,
    llm_node,
    retrieve_node,
    evaluate_node,
    web_search_node,
    termination
])

machine.connect(entry, prep_node)
machine.connect(prep_node, retrieve_node)
machine.connect(retrieve_node, evaluate_node)
machine.connect(evaluate_node, [web_search_node, llm_node], should_web_search)
machine.connect(web_search_node, llm_node)
machine.connect(llm_node, termination)

In [24]:
session_messages = []

def run_turn(query, turn_num):
    global session_messages
    print(f"\n--- Turn {turn_num} ---")
    print(f"Q: {query}")

    initial_state: AgentState = {
        "user_query": query,
        "instructions": "",
        "messages": session_messages,  # carry over previous messages
        "current_tool_calls": None,
        "session_id": "udaplay"
      }

    run = machine.run(initial_state)
    final_state = run.get_final_state()
    session_messages = final_state["messages"]  # save messages for next turn

    answer = session_messages[-1].content
    print(f"A: {answer}")
    print(f"[Message history length: {len(session_messages)}]")
    return answer

  # Turn 1: ask about a game
run_turn("When was Super Mario 64 released?", 1)

  # Turn 2: follow-up using pronoun (requires memory of turn 1)
run_turn("What platform was it released on?", 2)

  # Turn 3: another follow-up
run_turn("Was it a 3D platformer?", 3)

print("\n--- Memory Context Stored ---")
print(long_term_memory.get_context())


--- Turn 1 ---
Q: When was Super Mario 64 released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: prepare_messages
[StateMachine] Executing step: retrieve_game
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: llm
[StateMachine] Terminating: __termination__
A: **Super Mario 64** was released in **1996** for the Nintendo 64. This game is recognized as a groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.
[Message history length: 5]

--- Turn 2 ---
Q: What platform was it released on?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: prepare_messages
[StateMachine] Executing step: retrieve_game
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: game_web_search
[StateMachine] Executing step: llm
[StateMachine] Terminating: __termination__
A: **Super Mario 64** was released on the **Nintendo 64** platform.
[Message history length

In [25]:
from lib.messages import AIMessage, UserMessage
import json


def run_with_trace(query):
    print(f"\n{'='*60}")
    print(f"QUERY: {query}")
    print('='*60)

    initial_state: AgentState = {
        "user_query": query,
        "instructions": "",
        "messages": [],
        "current_tool_calls": None,
        "session_id": "udaplay"
    }

    run = machine.run(initial_state)
    final_state = run.get_final_state()
    messages = final_state["messages"]

    for msg in messages:
        if isinstance(msg, UserMessage) and msg.content.startswith("[Tool:"):
            header, _, content = msg.content.partition("\n")
            tool_name = header.replace("[Tool: ", "").replace("]", "").strip()

            if tool_name == "retrieve_game":
                print(f"\n[RETRIEVAL RESULTS]")
                try:
                    docs = eval(content)
                    for i, doc in enumerate(docs[:3]):
                        print(f"  {i+1}. {doc.get('Name')} | {doc.get('Platform')} | {doc.get('YearOfRelease')}")
                except:
                    print(f"  {content[:200]}")

            elif tool_name == "evaluate_retrieval":
                print(f"\n[EVALUATION DECISION]")
                try:
                    result = eval(content)
                    print(f"  Useful: {result.get('useful')}")
                    print(f"  Reason: {result.get('description', '')[:200]}")
                except:
                    print(f"  {content[:200]}")

            elif tool_name == "game_web_search":
                print(f"\n[WEB SEARCH SOURCES]")
                try:
                    result = eval(content)
                    for r in result.get("results", [])[:2]:
                        print(f"  - {r.get('title')}")
                        print(f"    URL: {r.get('url')}")
                except:
                    print(f"  {content[:200]}")

    print(f"\n[FINAL ANSWER]")
    print(f"  {messages[-1].content}")

    # Citations (if web search was used)
    for msg in messages:
        if isinstance(msg, UserMessage) and msg.content.startswith("[Tool: game_web_search]"):
            _, _, content = msg.content.partition("\n")
            try:
                result = eval(content)
                print(f"\n[CITATIONS]")
                for r in result.get("results", [])[:2]:
                    print(f"  - {r.get('title')}: {r.get('url')}")
            except:
                pass

    long_term_memory.save(query, messages[-1].content)


run_with_trace("When was Pokémon Gold and Silver released?")
run_with_trace("Which one was the first 3D platformer Mario game?")
run_with_trace("Was Mortal Kombat X released for PlayStation 5?")


QUERY: When was Pokémon Gold and Silver released?
[StateMachine] Starting: __entry__
[StateMachine] Executing step: prepare_messages
[StateMachine] Executing step: retrieve_game
[StateMachine] Executing step: evaluate_retrieval
[StateMachine] Executing step: llm
[StateMachine] Terminating: __termination__

[RETRIEVAL RESULTS]
  1. Pokémon Gold and Silver | Game Boy Color | 1999
  2. Pokémon Ruby and Sapphire | Game Boy Advance | 2002
  3. Super Mario 64 | Nintendo 64 | 1996

[EVALUATION DECISION]
  Useful: True
  Reason: The documents retrieved contain sufficient information to respond to the query regarding the release date of Pokémon Gold and Silver. The first document explicitly states the game title 'Pokémon Gold 

[FINAL ANSWER]
  Pokémon Gold and Silver were released in **1999** for the Game Boy Color. These games are part of the second generation of Pokémon games, introducing new regions, Pokémon, and gameplay mechanics.

QUERY: Which one was the first 3D platformer Mario game?

 ## Agent Demo - Tool Trace Report

  This section demonstrates the agent's full reasoning process for three queries.

  For each query, the trace shows:
  1. **Tool Called** - which tool the agent used (retrieve_game, evaluate_retrieval, game_web_search)
  2. **Retrieval Results** - top games found in the vector database with metadata
  3. **Evaluation Decision** - whether retrieved docs were useful and why
  4. **Web Search Sources** - URLs cited when web search was used as fallback
  5. **Final Answer** - the agent's final response

  The agent follows this workflow:
  - First searches the local ChromaDB vector database
  - Evaluates if results are good enough to answer the question
  - Falls back to Tavily web search if results are insufficient